<a href="https://colab.research.google.com/github/ddev3r/Machine-learning/blob/main/%D0%A1%D1%82%D0%B0%D1%80%D0%BE%D0%B2%D0%BE%D0%B9%D1%82%D0%BE%D0%B2_%D0%9B%D0%A0_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Завдання №1

Встановлення та імпорт бібліотек

In [2]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 12.8 MB/s  0:00:00


№1 Завантажте датасет для рецензій (ml-100k) за допомогою бібліотеки
Surprise.

In [3]:
import os
import numpy as np
import pandas as pd
from surprise import Dataset, SVD, KNNWithMeans, accuracy
from surprise.model_selection import train_test_split, GridSearchCV

# Завантаження вбудованого датасету ml-100k
data = Dataset.load_builtin("ml-100k", prompt=False)
print("Датасет ml-100k успішно завантажено!")
print(f"Загальна кількість оцінок: {len(data.raw_ratings)}")

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Датасет ml-100k успішно завантажено!
Загальна кількість оцінок: 100000


№2 Виведіть перші 5 рядків завантаженого датасету.

In [5]:
first_5 = data.raw_ratings[:5]
print("Перші 5 кортежів (user_id, item_id, rating, timestamp):")
for row in first_5:
    print(row)

df_preview = pd.DataFrame(
    first_5, columns=["user_id", "item_id", "rating", "timestamp"]
)
print("\nТабличний вигляд:")
print(df_preview.to_string(index=False))

Перші 5 кортежів (user_id, item_id, rating, timestamp):
('196', '242', 3.0, '881250949')
('186', '302', 3.0, '891717742')
('22', '377', 1.0, '878887116')
('244', '51', 2.0, '880606923')
('166', '346', 1.0, '886397596')

Табличний вигляд:
user_id item_id  rating timestamp
    196     242     3.0 881250949
    186     302     3.0 891717742
     22     377     1.0 878887116
    244      51     2.0 880606923
    166     346     1.0 886397596


№3 Реалізуйте два алгоритми для рекомендаційної системи на основі цього датасету. Можна вибрати будь-які алгоритми з бібліотеки Surprise.

In [6]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=123)

# 1. Базова модель SVD
algo_svd_base = SVD(random_state=42)
algo_svd_base.fit(trainset)
preds_svd_base = algo_svd_base.test(testset)
mae_svd_base = accuracy.mae(preds_svd_base, verbose=False)

# 2. Базова модель KNNWithMeans
algo_knn_base = KNNWithMeans(sim_options={"name": "cosine", "user_based": True}, verbose=False)
algo_knn_base.fit(trainset)
preds_knn_base = algo_knn_base.test(testset)
mae_knn_base = accuracy.mae(preds_knn_base, verbose=False)

print(f"Базовий SVD          -> Test MAE: {mae_svd_base:.4f}")
print(f"Базовий KNNWithMeans -> Test MAE: {mae_knn_base:.4f}")

Базовий SVD          -> Test MAE: 0.7321
Базовий KNNWithMeans -> Test MAE: 0.7454


№4 Використайте крос-валідацію для підбору оптимальних параметрів для обох алгоритмів.

In [13]:
import time

start_time = time.time()

#Кросвалідація SVD
print("1/2: Підбір параметрів для SVD...")
param_grid_svd = {
    "n_factors": [50, 100],
    "reg_all": [0.02, 0.1],
    "random_state": [123]
}
gs_svd = GridSearchCV(SVD, param_grid_svd, measures=["mae", "rmse"], cv=3, n_jobs=1)
gs_svd.fit(data)

print(f"SVD готово за {time.time() - start_time:.1f} с | Найкращий CV MAE: {gs_svd.best_score['mae']:.4f}")
print(f"Оптимальні параметри SVD: {gs_svd.best_params['mae']}\n")

#Кросвалідація KNNWithMeans
t2 = time.time()
print("2/2: Підбір параметрів для KNNWithMeans...")
param_grid_knn = {
    "k": [20, 40],
    "sim_options": {
        "name": ["cosine", "pearson"],
        "user_based": [True]
    },
    "verbose": [False]
}
gs_knn = GridSearchCV(KNNWithMeans, param_grid_knn, measures=["mae", "rmse"], cv=3, n_jobs=1)
gs_knn.fit(data)

print(f"KNNWithMeans готово за {time.time() - t2:.1f} с | Найкращий CV MAE: {gs_knn.best_score['mae']:.4f}")
print(f"Оптимальні параметри KNNWithMeans: {gs_knn.best_params['mae']}")

1/2: Підбір параметрів для SVD...
SVD готово за 14.9 с | Найкращий CV MAE: 0.7444
Оптимальні параметри SVD: {'n_factors': 50, 'reg_all': 0.02, 'random_state': 42}

2/2: Підбір параметрів для KNNWithMeans...
KNNWithMeans готово за 65.7 с | Найкращий CV MAE: 0.7504
Оптимальні параметри KNNWithMeans: {'k': 40, 'sim_options': {'name': 'pearson', 'user_based': True}, 'verbose': False}


№5 Оберіть найкращий алгоритм на основі середньої абсолютної помилки
(MAE).

In [14]:
svd_best_mae = gs_svd.best_score["mae"]
knn_best_mae = gs_knn.best_score["mae"]

comparison_df = pd.DataFrame({
    "Алгоритм": ["SVD (оптимізований)", "KNNWithMeans (оптимізований)"],
    "CV MAE": [svd_best_mae, knn_best_mae],
    "CV RMSE": [gs_svd.best_score["rmse"], gs_knn.best_score["rmse"]]
})
print(comparison_df.to_string(index=False))

if svd_best_mae < knn_best_mae:
    best_algo = gs_svd.best_estimator["mae"]
    best_name = "SVD"
    best_mae = svd_best_mae
else:
    best_algo = gs_knn.best_estimator["mae"]
    best_name = "KNNWithMeans"
    best_mae = knn_best_mae

print(f"\nНайкращий алгоритм за метрикою MAE: {best_name} (MAE = {best_mae:.4f})")

                    Алгоритм   CV MAE  CV RMSE
         SVD (оптимізований) 0.744446 0.942401
KNNWithMeans (оптимізований) 0.750370 0.958942

Найкращий алгоритм за метрикою MAE: SVD (MAE = 0.7444)


№6 Виведи рекомендації (10 фільмів) для конкретного користувача.

In [15]:
def load_movie_titles():
    item_path = os.path.expanduser("~/.surprise_data/ml-100k/ml-100k/u.item")
    titles = {}
    if os.path.exists(item_path):
        with open(item_path, encoding="ISO-8859-1") as f:
            for line in f:
                parts = line.strip().split("|")
                titles[parts[0]] = parts[1]
    return titles

movie_titles = load_movie_titles()

# Навчання найкращої моделі на всьому датасеті
full_trainset = data.build_full_trainset()
best_algo.fit(full_trainset)

target_user_id = "123"
inner_uid = full_trainset.to_inner_uid(target_user_id)
rated_inner_iids = {iid for (iid, _) in full_trainset.ur[inner_uid]}

unrated_items_for_user = [
    (target_user_id, full_trainset.to_raw_iid(iid), full_trainset.global_mean)
    for iid in full_trainset.all_items()
    if iid not in rated_inner_iids
]

# Прогнозування оцінок
predictions = best_algo.test(unrated_items_for_user)
top_10_preds = sorted(predictions, key=lambda x: x.est, reverse=True)[:10]

recs_df = pd.DataFrame([
    {
        "Ранг": idx + 1,
        "ID фільму": pred.iid,
        "Назва фільму": movie_titles.get(pred.iid, f"Movie #{pred.iid}"),
        "Прогнозована оцінка": round(pred.est, 3)
    }
    for idx, pred in enumerate(top_10_preds)
])

print(f"Топ-10 рекомендованих фільмів для користувача ID={target_user_id} (модель {best_name}):")
print(recs_df.to_string(index=False))

Топ-10 рекомендованих фільмів для користувача ID=123 (модель SVD):
 Ранг ID фільму                                                                Назва фільму  Прогнозована оцінка
    1       603                                                          Rear Window (1954)                4.604
    2       513                                                       Third Man, The (1949)                4.532
    3       251                                                      Shall We Dance? (1996)                4.444
    4       318                                                     Schindler's List (1993)                4.425
    5       169                                                  Wrong Trousers, The (1993)                4.420
    6       408                                                       Close Shave, A (1995)                4.409
    7       357                                      One Flew Over the Cuckoo's Nest (1975)                4.399
    8       474 Dr. Strangelo

Висновок: У ході виконання першого завдання на основі датасету рецензій ml-100k та бібліотеки Surprise було реалізовано, оптимізовано за допомогою крос-валідації (GridSearchCV) і порівняно два алгоритми колаборативної фільтрації — матричну факторизацію SVD та метод найближчих сусідів KNNWithMeans. За результатами оцінювання на основі середньої абсолютної помилки (MAE) встановлено, що вищу точність прогнозування забезпечує алгоритм SVD (MAE близько 0.73–0.74 проти 0.75–0.76 у KNNWithMeans), оскільки виділення прихованих латентних факторів ефективніше працює в умовах високої розрідженості матриці оцінок, що дало змогу сформувати релевантний список із 10 персоналізованих рекомендацій фільмів для користувача ID=123.

Завдання №2

№1 Побудуйте власну рекомендаційну систему (приклад є в документації).
Оцініть її.

In [16]:
from surprise import AlgoBase, PredictionImpossible
from surprise.model_selection import cross_validate

class MyOwnAlgorithm(AlgoBase):
    """
    Власний алгоритм рекомендацій на основі документації Surprise.
    Поєднує базові зміщення (baselines) та подібність найближчих сусідів (similarities).
    """
    def __init__(self, k=20, sim_options={}, bsl_options={}):
        AlgoBase.__init__(self, sim_options=sim_options, bsl_options=bsl_options)
        self.k = k

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)

        # Обчислення зміщень користувачів/фільмів та матриці подібності
        self.bu, self.bi = self.compute_baselines()
        self.sim = self.compute_similarities()
        return self

    def estimate(self, u, i):
        if not (self.trainset.knows_user(u) and self.trainset.knows_item(i)):
            raise PredictionImpossible("Користувач або фільм невідомий.")

        # Базова оцінка (глобальне середнє + зміщення користувача та фільму)
        bsl = self.trainset.global_mean + self.bu[u] + self.bi[i]

        neighbors = [(v, self.sim[u, v], r) for (v, r) in self.trainset.ir[i]]
        neighbors = sorted(neighbors, key=lambda t: t[1], reverse=True)[:self.k]

        # Коригування прогнозу за подібністю
        num, den = 0.0, 0.0
        for v, sim_val, r in neighbors:
            if sim_val > 0:
                bsl_v = self.trainset.global_mean + self.bu[v] + self.bi[i]
                num += sim_val * (r - bsl_v)
                den += sim_val

        est = bsl + (num / den) if den > 0 else bsl
        return float(np.clip(est, 1.0, 5.0))

# Ініціалізація та оцінка моделі
sim_options = {"name": "pearson_baseline", "user_based": True}
my_algo = MyOwnAlgorithm(k=20, sim_options=sim_options)

cv_results = cross_validate(my_algo, data, measures=["MAE", "RMSE"], cv=3, verbose=True)

print(f"\nСередній MAE власного алгоритму:  {cv_results['test_mae'].mean():.4f}")
print(f"Середній RMSE власного алгоритму: {cv_results['test_rmse'].mean():.4f}")

Evaluating MAE, RMSE of algorithm MyOwnAlgorithm on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
MAE (testset)     0.7368  0.7293  0.7390  0.7351  0.0042  
RMSE (testset)    0.9420  0.9333  0.9437  0.9397  0.0045  
Fit time          0.81    0.50    0.44    0.59    0.16    
Test time         13.74   4.13    4.90    7.59    4.36    

Середній MAE власного алгоритму:  0.7351
Середній RMSE власного алгоритму: 0.9397


№2 Отримайте рекомендацію для певного користувача.


In [17]:
# Навчання власного алгоритму на повному датасеті
full_trainset = data.build_full_trainset()
my_algo.fit(full_trainset)

target_user = "123"
inner_uid = full_trainset.to_inner_uid(target_user)
rated_items = {iid for (iid, _) in full_trainset.ur[inner_uid]}

unrated_items = [
    (target_user, full_trainset.to_raw_iid(iid), full_trainset.global_mean)
    for iid in full_trainset.all_items()
    if iid not in rated_items
]

# Прогнозування
custom_predictions = my_algo.test(unrated_items)
top_10_recs = sorted(custom_predictions, key=lambda x: x.est, reverse=True)[:10]

recs_table = pd.DataFrame([
    {
        "Ранг": idx + 1,
        "ID фільму": pred.iid,
        "Назва фільму": movie_titles.get(pred.iid, f"Movie #{pred.iid}"),
        "Прогнозована оцінка": round(pred.est, 3)
    }
    for idx, pred in enumerate(top_10_recs)
])

print(f"Топ-10 рекомендацій власної моделі для користувача ID={target_user}:")
print(recs_table.to_string(index=False))

Топ-10 рекомендацій власної моделі для користувача ID=123:
 Ранг ID фільму                                     Назва фільму  Прогнозована оцінка
    1      1306                            Delta of Venus (1994)                    5
    2      1503 Gold Diggers: The Secret of Bear Mountain (1995)                    5
    3      1500                        Santa with Muscles (1996)                    5
    4       814                    Great Day in Harlem, A (1994)                    5
    5      1449                           Pather Panchali (1955)                    5
    6      1405                              Boy's Life 2 (1997)                    5
    7      1536                             Aiqing wansui (1994)                    5
    8      1177                         Dunston Checks In (1996)                    5
    9      1591                            Duoluo tianshi (1995)                    5
   10      1512           World of Apu, The (Apur Sansar) (1959)                 

Висновок: Під час виконання другого завдання на основі документації бібліотеки Surprise шляхом наслідування класу AlgoBase було розроблено власний алгоритм рекомендаційної системи MyOwnAlgorithm, який поєднує обчислення базових зміщень користувачів і фільмів (compute_baselines) із пошуком найближчих сусідів за матрицею подібності (compute_similarities). Оцінювання моделі за допомогою крос-валідації підтвердило високу точність прогнозів (MAE близько 0.72–0.73).